# **Loading V1.1 Datasets for V2.1**


In [75]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [76]:
from pathlib import Path

# Input is V1.1; output models and predictions belong to V2.1.
BASE_PATH = Path("/content/drive/MyDrive/Projects/Tennis/v1_1/datasets")
PROJECT_DIR = BASE_PATH.parent.parent / "v2_1"


In [77]:
import pandas as pd

train = pd.read_parquet(BASE_PATH / "v1_1_train.parquet")
val = pd.read_parquet(BASE_PATH / "v1_1_val.parquet")
train_log = pd.read_parquet(BASE_PATH / "v1_1_train_log.parquet")
val_log = pd.read_parquet(BASE_PATH / "v1_1_val_log.parquet")
# Test features are loaded only in the final prediction section.


In [78]:
# V1.1 partitions and row order are authoritative; never re-split or join on player names.
import json
import numpy as np
from sklearn.model_selection import TimeSeriesSplit

match_metadata = pd.read_csv(BASE_PATH / "matches_v1_1_metadata.csv")
manifest = json.loads((BASE_PATH / "preprocessing_manifest.json").read_text())
required_meta = {"dataset_row", "split", "tourney_date", "playerA_wins"}
if not required_meta.issubset(match_metadata.columns):
    raise ValueError(f"Missing metadata columns: {required_meta - set(match_metadata.columns)}")
if match_metadata["dataset_row"].isna().any() or not match_metadata["dataset_row"].is_unique:
    raise ValueError("dataset_row must be a unique, non-missing V1.1 row identifier")
if not match_metadata["dataset_row"].is_monotonic_increasing:
    raise ValueError("V1.1 metadata row identifiers are out of order")
if not match_metadata["split"].isin(["train", "val", "test"]).all():
    raise ValueError("Unexpected V1.1 split label")
metadata_by_split = {
    split: match_metadata.loc[match_metadata["split"].eq(split)].reset_index(drop=True)
    for split in ("train", "val", "test")
}
TARGETS_V1 = ["playerA_match_p", "playerB_match_p"]
for split, normal, logged in (("train", train, train_log), ("val", val, val_log)):
    metadata = metadata_by_split[split]
    if not (len(normal) == len(logged) == len(metadata) == manifest["split_sizes"][split]):
        raise ValueError(f"{split}: V1.1 Parquet/metadata/manifest lengths differ")
    if not normal.columns.equals(logged.columns):
        raise ValueError(f"{split}: normal/log column schema differs")
    if not set(TARGETS_V1).issubset(normal.columns):
        raise ValueError(f"{split}: missing V1.1 regression targets")
    if not normal[TARGETS_V1].reset_index(drop=True).equals(logged[TARGETS_V1].reset_index(drop=True)):
        raise ValueError(f"{split}: normal/log regression targets are not row-aligned")
    if {"playerA_wins", "score", "minutes", "match_id"}.intersection(normal.columns):
        raise ValueError(f"{split}: outcome or identifier leaked into features")
    if not metadata["playerA_wins"].isin([0, 1]).all():
        raise ValueError(f"{split}: invalid classification labels")
    d = pd.to_datetime(metadata["tourney_date"], errors="coerce")
    if d.isna().any() or not d.is_monotonic_increasing:
        raise ValueError(f"{split}: missing or nonchronological tournament dates")
    if split == "val" and not normal.columns.equals(train.columns):
        raise ValueError("Validation feature schema/order differs from training")
if manifest.get("targets") != TARGETS_V1:
    raise ValueError("Unexpected V1.1 target declaration in preprocessing manifest")
if not (pd.to_datetime(metadata_by_split["train"]["tourney_date"]).max() <
        pd.to_datetime(metadata_by_split["val"]["tourney_date"]).min() <
        pd.to_datetime(metadata_by_split["test"]["tourney_date"]).min()):
    raise ValueError("V1.1 chronological partition boundaries overlap")

train_dates = pd.to_datetime(metadata_by_split["train"]["tourney_date"]).reset_index(drop=True)

class DateGroupedCV:
    """Expanding-time CV on unique V1.1 tournament reference dates."""
    def __init__(self, n_splits=3):
        self.n_splits = n_splits
    def split(self, X, y=None, groups=None):
        if len(X) != len(train_dates):
            raise ValueError("CV inputs must use the original V1.1 training partition")
        dates = train_dates
        unique_dates = np.sort(dates.unique())
        for tr, va in TimeSeriesSplit(n_splits=self.n_splits).split(unique_dates):
            tr_idx = np.flatnonzero(dates.isin(unique_dates[tr]).to_numpy())
            va_idx = np.flatnonzero(dates.isin(unique_dates[va]).to_numpy())
            yield tr_idx, va_idx
    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

tscv = DateGroupedCV(n_splits=3)
for subdir in ("models/base_models", "models/stage1_models",
               "models/stage2_models", "models/final_models", "predictions"):
    (PROJECT_DIR / subdir).mkdir(parents=True, exist_ok=True)
print("V2.1 data preflight passed:",
      {split: len(meta) for split, meta in metadata_by_split.items()})


V2.1 data preflight passed: {'train': 181632, 'val': 26109, 'test': 51666}


# **Seperating Predictors and Labels**

In [79]:
import gc

# V1.1 service targets are post-match outcomes: never use them as classifier features.
X_train = train.drop(columns=TARGETS_V1).reset_index(drop=True)
X_val = val.drop(columns=TARGETS_V1).reset_index(drop=True)
y_train = metadata_by_split["train"]["playerA_wins"].astype(int).reset_index(drop=True)
y_val = metadata_by_split["val"]["playerA_wins"].astype(int).reset_index(drop=True)

if not X_train.columns.equals(X_val.columns):
    raise ValueError("V2.1 training/validation predictors differ")
if X_train.columns.tolist() != manifest["feature_columns"]:
    raise ValueError("V2.1 predictors do not match the V1.1 preprocessing manifest")
del train, val
gc.collect()


8

In [80]:
# Apply the same V1.1 log feature variant to linear classifiers only.
X_train_log = train_log.drop(columns=TARGETS_V1).reset_index(drop=True)
X_val_log = val_log.drop(columns=TARGETS_V1).reset_index(drop=True)
if not (X_train.columns.equals(X_train_log.columns) and X_val.columns.equals(X_val_log.columns)):
    raise ValueError("V1.1 log/normal feature orders differ")
del train_log, val_log
gc.collect()


0

# **Training Models**

## Importing Libraries

In [81]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
print("V2.1 uses training-fitted sklearn Pipelines for linear models.")


V2.1 uses training-fitted sklearn Pipelines for linear models.


## Logistic Regression

### Base Model

In [12]:
from sklearn.linear_model import LogisticRegression
import time

start_time = time.time()
log = Pipeline([("scaler", StandardScaler()),
                ("model", LogisticRegression(random_state=42, max_iter=1000))])
log.fit(X_train_log, y_train)
train_time = time.time() - start_time
MODEL_PATH = PROJECT_DIR / "models/base_models/logReg_base_v2_1.pkl"
joblib.dump(log, MODEL_PATH)
print("Model Saved. Training time:", train_time)


Model Saved. Training time: 17.654971599578857


In [13]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, brier_score_loss
import gc

val_preds = log.predict(X_val_log)
val_prob = log.predict_proba(X_val_log)[:, 1]

print(f"Accuracy: {accuracy_score(y_val, val_preds)}")
print(f"ROC-AUC: {roc_auc_score(y_val, val_prob)}")
print(f"Log Loss: {log_loss(y_val, val_prob)}")
print(f"Brier Score: {brier_score_loss(y_val, val_prob)}")

del log
gc.collect()

Accuracy: 0.6590064728637635
ROC-AUC: 0.7254557904085239
Log Loss: 0.608726713801426
Brier Score: 0.21106579814747473


27

### Hyperparameter Tuning

In [14]:
from scipy.stats import loguniform
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
import time

param_grid = {
    "model__C": loguniform(1e-4, 100),
    "model__penalty": ["l2"],
    "model__class_weight": [None, "balanced"],
}
start_time = time.time()
logreg = Pipeline([("scaler", StandardScaler()),
                   ("model", LogisticRegression(max_iter=1000, random_state=42))])
rscv = RandomizedSearchCV(logreg, param_distributions=param_grid, n_iter=20,
                          scoring="neg_log_loss", cv=tscv, random_state=42,
                          verbose=2, error_score="raise")
search = rscv.fit(X_train_log, y_train)
print(f"RandomizedSearchCV completed in {time.time() - start_time:.2f} seconds.")


Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END model__C=0.017670169402947963, model__class_weight=None, model__penalty=l2; total time=   1.8s
[CV] END model__C=0.017670169402947963, model__class_weight=None, model__penalty=l2; total time=   4.1s
[CV] END model__C=0.017670169402947963, model__class_weight=None, model__penalty=l2; total time=   9.2s
[CV] END model__C=0.0012606912518374083, model__class_weight=balanced, model__penalty=l2; total time=   0.9s
[CV] END model__C=0.0012606912518374083, model__class_weight=balanced, model__penalty=l2; total time=   2.1s
[CV] END model__C=0.0012606912518374083, model__class_weight=balanced, model__penalty=l2; total time=   3.9s
[CV] END model__C=0.39079671568228835, model__class_weight=None, model__penalty=l2; total time=   5.9s
[CV] END model__C=0.39079671568228835, model__class_weight=None, model__penalty=l2; total time=   6.9s
[CV] END model__C=0.39079671568228835, model__class_weight=None, model__penalty=l2; total time

In [15]:
search.best_params_

{'model__C': np.float64(0.0012606912518374083),
 'model__class_weight': 'balanced',
 'model__penalty': 'l2'}

In [16]:
search.best_score_

np.float64(-0.5912392964656189)

In [17]:
search.cv_results_

{'mean_fit_time': array([4.78403401, 2.07266919, 7.92094421, 6.30738576, 1.58917618,
        4.22137252, 7.92112851, 1.63766464, 8.84170222, 1.17440089,
        2.05632242, 8.45804365, 5.68557819, 7.94898216, 1.84124271,
        9.9561189 , 6.52086806, 7.90310907, 7.46176227, 5.83282153]),
 'std_fit_time': array([3.07353512, 1.11792674, 2.35435886, 2.67135706, 0.70272754,
        2.35448714, 2.45355945, 0.82679465, 3.1694208 , 0.50486721,
        1.08164158, 2.07316946, 2.65941486, 2.8407629 , 1.08338024,
        2.75882634, 2.83183314, 2.45018081, 2.90916689, 2.35742448]),
 'mean_score_time': array([0.22306403, 0.23727194, 0.18254232, 0.22470927, 0.24385532,
        0.22565707, 0.22665517, 0.24825144, 0.23069112, 0.21205711,
        0.19349202, 0.19347652, 0.2084175 , 0.22441077, 0.19074655,
        0.24315564, 0.1942114 , 0.2398049 , 0.18340158, 0.22491384]),
 'std_score_time': array([0.05400669, 0.0913887 , 0.04119265, 0.09508998, 0.04802269,
        0.08153647, 0.05801113, 0.040828

### Best Model

In [18]:
# The fitted Pipeline includes its training-only StandardScaler.
logreg = search.best_estimator_
PATH = PROJECT_DIR / "models/stage1_models/logReg_stage1_v2_1.pkl"
joblib.dump(logreg, PATH)
print("Model Saved.")


Model Saved.


In [19]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, brier_score_loss
import gc

val_preds = logreg.predict(X_val_log)
val_prob = logreg.predict_proba(X_val_log)[:, 1]

print(f"Accuracy: {accuracy_score(y_val, val_preds)}")
print(f"ROC-AUC: {roc_auc_score(y_val, val_prob)}")
print(f"Log Loss: {log_loss(y_val, val_prob)}")
print(f"Brier Score: {brier_score_loss(y_val, val_prob)}")

del logreg
gc.collect()

Accuracy: 0.6584319583285457
ROC-AUC: 0.7253227160008942
Log Loss: 0.6088086433713008
Brier Score: 0.21110442807300803


396

## SGD Classifier

### Base Model

In [20]:
from sklearn.linear_model import SGDClassifier
import time

start_time = time.time()
sgd = Pipeline([("scaler", StandardScaler()),
                ("model", SGDClassifier(loss="log_loss", penalty="l2", alpha=0.0001,
                                        learning_rate="optimal", max_iter=2000, tol=1e-3,
                                        early_stopping=True, validation_fraction=0.1,
                                        n_iter_no_change=10, random_state=42))])
sgd.fit(X_train_log, y_train)
elapsed_time = time.time() - start_time
MODEL_PATH = PROJECT_DIR / "models/base_models/sgd_base_v2_1.pkl"
joblib.dump(sgd, MODEL_PATH)
print("Model Saved. Training time:", elapsed_time)


Model Saved. Training time: 7.9430341720581055


In [21]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, brier_score_loss
import gc

val_preds = sgd.predict(X_val_log)
val_prob = sgd.predict_proba(X_val_log)[:, 1]

print(f"Accuracy: {accuracy_score(y_val, val_preds)}")
print(f"ROC-AUC: {roc_auc_score(y_val, val_prob)}")
print(f"Log Loss: {log_loss(y_val, val_prob)}")
print(f"Brier Score: {brier_score_loss(y_val, val_prob)}")

del sgd
gc.collect()

Accuracy: 0.6495844344861925
ROC-AUC: 0.7120434885614984
Log Loss: 0.6241051761560338
Brier Score: 0.21697512756954437


0

### Hyperparameter Tuning

In [23]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [optuna]2m2/3 [optuna]]


In [24]:
import optuna
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import log_loss
from sklearn.model_selection import TimeSeriesSplit
import time
import warnings

warnings.filterwarnings("ignore")

start_time = time.time()

y_train = y_train.squeeze()

def objective(trial):

    params = {

        "loss": trial.suggest_categorical(
            "loss",
            ["log_loss", "modified_huber"]
        ),

        "penalty": trial.suggest_categorical(
            "penalty",
            ["l2", "l1", "elasticnet"]
        ),

        "alpha": trial.suggest_float(
            "alpha",
            1e-6,
            1e-2,
            log=True
        ),

        "learning_rate": trial.suggest_categorical(
            "learning_rate",
            ["optimal", "adaptive", "invscaling"]
        ),

        "average": trial.suggest_categorical(
            "average",
            [False, True]
        )

    }

    # Conditional parameters

    if params["learning_rate"] in ["adaptive", "invscaling"]:

        params["eta0"] = trial.suggest_float(
            "eta0",
            1e-4,
            1e-1,
            log=True
        )

    if params["learning_rate"] == "invscaling":

        params["power_t"] = trial.suggest_float(
            "power_t",
            0.1,
            0.9
        )

    if params["penalty"] == "elasticnet":

        params["l1_ratio"] = trial.suggest_float(
            "l1_ratio",
            0.0,
            1.0
        )

    scores = []

    tscv = DateGroupedCV(n_splits=3)

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train_log)):

        X_fold_train = X_train_log.iloc[train_idx]
        X_fold_val = X_train_log.iloc[val_idx]

        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        model = Pipeline([("scaler", StandardScaler()), ("model", SGDClassifier(

            max_iter=3000,
            tol=1e-3,

            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=10,

            random_state=42,

            **params

        ))])

        model.fit(X_fold_train, y_fold_train)

        probs = model.predict_proba(X_fold_val)

        score = log_loss(y_fold_val, probs)

        scores.append(score)

        trial.report(sum(scores) / (fold + 1), fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return sum(scores) / len(scores)


study = optuna.create_study(
    direction="minimize",
    study_name="SGD Classifier",
    pruner=optuna.pruners.MedianPruner()
)

study.optimize(
    objective,
    n_trials=25,
    show_progress_bar=True
)

print("Elapsed Time:", time.time() - start_time)
print("Best Params:", study.best_params)
print("Best CV Log Loss:", study.best_value)

[I 2026-11-20 09:09:19,249] A new study created in memory with name: SGD Classifier


  0%|          | 0/25 [00:00<?, ?it/s]

[I 2026-11-20 09:09:42,911] Trial 0 finished with value: 0.592701147326372 and parameters: {'loss': 'log_loss', 'penalty': 'l2', 'alpha': 0.006170327855797607, 'learning_rate': 'adaptive', 'average': True, 'eta0': 0.00015251232234638458}. Best is trial 0 with value: 0.592701147326372.
[I 2026-11-20 09:11:00,993] Trial 1 finished with value: 11.165626938889114 and parameters: {'loss': 'modified_huber', 'penalty': 'l1', 'alpha': 5.9876766115563635e-05, 'learning_rate': 'adaptive', 'average': True, 'eta0': 0.011189523552346637}. Best is trial 0 with value: 0.592701147326372.
[I 2026-11-20 09:11:25,001] Trial 2 finished with value: 5.5220536021953315 and parameters: {'loss': 'log_loss', 'penalty': 'l1', 'alpha': 1.2227508131743903e-05, 'learning_rate': 'optimal', 'average': True}. Best is trial 0 with value: 0.592701147326372.
[I 2026-11-20 09:11:35,270] Trial 3 finished with value: 0.6191464774584251 and parameters: {'loss': 'modified_huber', 'penalty': 'l1', 'alpha': 1.608538264153464e-0

In [25]:
best_params = study.best_params
print(best_params)

{'loss': 'log_loss', 'penalty': 'l2', 'alpha': 3.036144964403823e-05, 'learning_rate': 'invscaling', 'average': True, 'eta0': 0.0001082850788620255, 'power_t': 0.11310071075643513}


In [26]:
study.best_value

0.5922766995023164

### Best Model

In [27]:
from sklearn.linear_model import SGDClassifier

sgd = Pipeline([("scaler", StandardScaler()),
                ("model", SGDClassifier(**best_params, max_iter=3000, tol=1e-3,
                                        early_stopping=True, validation_fraction=0.1,
                                        n_iter_no_change=10, random_state=42))])
sgd.fit(X_train_log, y_train)
PATH = PROJECT_DIR / "models/stage1_models/sgd_stage1_v2_1.pkl"
joblib.dump(sgd, PATH)
print("Model Saved.")


Model Saved.


In [28]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, brier_score_loss
import gc

val_preds = sgd.predict(X_val_log)
val_prob = sgd.predict_proba(X_val_log)[:, 1]

print(f"Accuracy: {accuracy_score(y_val, val_preds)}")
print(f"ROC-AUC: {roc_auc_score(y_val, val_prob)}")
print(f"Log Loss: {log_loss(y_val, val_prob)}")
print(f"Brier Score: {brier_score_loss(y_val, val_prob)}")

del sgd
gc.collect()

Accuracy: 0.6583170554215022
ROC-AUC: 0.725042290351974
Log Loss: 0.6089423428516051
Brier Score: 0.2111683040069084


604

## Random Forest

### Base Model

In [29]:
# CPU sklearn RF avoids RAPIDS-only checkpoint conversion without changing model type.
from sklearn.ensemble import RandomForestClassifier
import joblib
import time

start_time = time.time()
rf = RandomForestClassifier(n_estimators=300, criterion="gini", min_samples_split=2,
                            min_samples_leaf=1, max_features="sqrt", bootstrap=True,
                            random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
elapsed_time = time.time() - start_time
MODEL_PATH = PROJECT_DIR / "models/base_models/randomForest_base_v2_1.pkl"
joblib.dump(rf, MODEL_PATH)
print("Model Saved. Training time:", elapsed_time)


Model Saved. Training time: 612.8155009746552


In [30]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, brier_score_loss
import gc

val_preds = rf.predict(X_val)
val_prob = rf.predict_proba(X_val)[:, 1]

print(f"Accuracy: {accuracy_score(y_val, val_preds)}")
print(f"ROC-AUC: {roc_auc_score(y_val, val_prob)}")
print(f"Log Loss: {log_loss(y_val, val_prob)}")
print(f"Brier Score: {brier_score_loss(y_val, val_prob)}")

del rf
gc.collect()

Accuracy: 0.6582404534834732
ROC-AUC: 0.7177986134137289
Log Loss: 0.6150855718946254
Brier Score: 0.213727141768909


48

### Hyperparameter Tuning

In [10]:
import time
# from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
import warnings
# import cudf

warnings.filterwarnings("ignore")

start_time = time.time()


param_distributions = {
    "n_estimators": [200, 500, 800],
    "criterion": ["gini", "entropy"],
    "max_depth": [10, 20, 30],
    "min_samples_split": [2, 6, 10, 14, 18],
    "min_samples_leaf": [3, 5, 7],
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [True, False]
}

tscv = DateGroupedCV(n_splits=3)


rf = RandomForestClassifier(random_state=42, n_jobs= -1)


random_search = RandomizedSearchCV(estimator=rf, param_distributions=param_distributions, scoring='neg_log_loss', cv=tscv, n_iter = 8,
                                        n_jobs=-1, random_state=42, verbose=1)

random_search.fit(X_train, y_train)

elapsed_time = time.time() - start_time
print(f"\nSearch Time: {elapsed_time/60:.2f} minutes")

Fitting 3 folds for each of 8 candidates, totalling 24 fits

Search Time: 204.23 minutes


In [11]:
random_search.best_score_

np.float64(-0.5957529580960358)

In [12]:
random_search.best_params_

{'n_estimators': 800,
 'min_samples_split': 6,
 'min_samples_leaf': 7,
 'max_features': 'sqrt',
 'max_depth': 30,
 'criterion': 'entropy',
 'bootstrap': True}

### Best Model

In [13]:
from sklearn.ensemble import RandomForestClassifier
import joblib
from google.colab import drive

rf = random_search.best_estimator_

#saving the models
drive.mount('/content/drive')
PATH = "/content/drive/MyDrive/Projects/Tennis/v2_1/models/stage1_models/randomForest_stage1_v2_1.pkl"
joblib.dump(rf, PATH)
print("Model Saved.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model Saved.


In [14]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, brier_score_loss
import gc

val_preds = rf.predict(X_val)
val_prob = rf.predict_proba(X_val)[:, 1]

print(f"Accuracy: {accuracy_score(y_val, val_preds)}")
print(f"ROC-AUC: {roc_auc_score(y_val, val_prob)}")
print(f"Log Loss: {log_loss(y_val, val_prob)}")
print(f"Brier Score: {brier_score_loss(y_val, val_prob)}")

del rf
gc.collect()

Accuracy: 0.6559040943735877
ROC-AUC: 0.7222634540203627
Log Loss: 0.6105969234947792
Brier Score: 0.21197437588041634


50

## XGBoost

### Base Model

In [15]:
from xgboost import XGBClassifier
import time

start_time = time.time()

xgb = XGBClassifier(objective="binary:logistic", eval_metric="logloss", n_estimators=500, learning_rate=0.05, max_depth=6, 
                    min_child_weight=1, subsample=0.8, colsample_bytree=0.8, gamma=0, reg_alpha=0, reg_lambda=1, random_state=42, 
                    tree_method="hist", device="cuda")

xgb.fit(X_train, y_train)
elapsed_time = time.time() - start_time

MODEL_PATH = "/content/drive/MyDrive/Projects/Tennis/v2_1/models/base_models/XGBoost_base_v2_1.pkl"
joblib.dump(xgb, MODEL_PATH)
print("Model Saved.")
print(f"Training time: {elapsed_time}")

Model Saved.
Training time: 29.882630348205566


In [16]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, brier_score_loss
import gc

val_preds = xgb.predict(X_val)
val_prob = xgb.predict_proba(X_val)[:, 1]

print(f"Accuracy: {accuracy_score(y_val, val_preds)}")
print(f"ROC-AUC: {roc_auc_score(y_val, val_prob)}")
print(f"Log Loss: {log_loss(y_val, val_prob)}")
print(f"Brier Score: {brier_score_loss(y_val, val_prob)}")

del xgb
gc.collect()

Accuracy: 0.6601938029032135
ROC-AUC: 0.7253699474986901
Log Loss: 0.609248081229493
Brier Score: 0.21127771381786825


50

### Hyperparameter Search

In [17]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [optuna]2m2/3 [optuna]]


In [18]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import log_loss
import time

start_time = time.time()

def objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.30, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 800, step=100),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
    }

    scores = []

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]

        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        model = XGBClassifier(objective="binary:logistic", eval_metric="logloss", tree_method="hist", device="cuda", random_state=42, 
        n_jobs=1, **params)

        model.fit(X_fold_train, y_fold_train, verbose=False)

        probs = model.predict_proba(X_fold_val)
        score = log_loss(y_fold_val, probs)

        scores.append(score)

        trial.report(sum(scores) / (fold + 1), fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return sum(scores) / len(scores)


study = optuna.create_study(direction="minimize", study_name="XGBoost Classifier GPU", pruner=optuna.pruners.MedianPruner())

study.optimize(objective, n_trials=30, show_progress_bar=True)

print("Elapsed Time GPU:", time.time() - start_time)

[I 2026-11-21 05:07:50,294] A new study created in memory with name: XGBoost Classifier GPU


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-11-21 05:10:59,933] Trial 0 finished with value: 0.5909374128838109 and parameters: {'learning_rate': 0.010169973109622822, 'n_estimators': 500, 'max_depth': 8, 'min_child_weight': 8, 'subsample': 0.7983113739701341, 'colsample_bytree': 0.8356113310646212, 'gamma': 4.788706363342895}. Best is trial 0 with value: 0.5909374128838109.
[I 2026-11-21 05:11:55,701] Trial 1 finished with value: 0.5941312447958537 and parameters: {'learning_rate': 0.09984959944272384, 'n_estimators': 500, 'max_depth': 3, 'min_child_weight': 6, 'subsample': 0.7184455142555781, 'colsample_bytree': 0.8484393030163866, 'gamma': 3.8726844615772116}. Best is trial 0 with value: 0.5909374128838109.
[I 2026-11-21 05:13:05,645] Trial 2 finished with value: 0.590963333573059 and parameters: {'learning_rate': 0.03408381097944505, 'n_estimators': 400, 'max_depth': 5, 'min_child_weight': 4, 'subsample': 0.6686073942108012, 'colsample_bytree': 0.7881328554101972, 'gamma': 3.308648454462315}. Best is trial 0 with val

In [19]:
study.best_value

0.5895455060471027

In [20]:
best_params = study.best_params
print(best_params)

{'learning_rate': 0.013323791833668426, 'n_estimators': 800, 'max_depth': 6, 'min_child_weight': 9, 'subsample': 0.7788228870366763, 'colsample_bytree': 0.771146474226091, 'gamma': 0.8957569290191403}


### Best Model

In [21]:
from xgboost import XGBClassifier
import joblib
from google.colab import drive
import time

start_time = time.time()

xgbc = XGBClassifier(objective="binary:logistic", eval_metric="logloss", tree_method="hist", device="cuda", random_state=42, 
                  **study.best_params)

xgbc.fit(X_train, y_train)
elapsed_time = time.time() - start_time

#saving the models
drive.mount('/content/drive')
PATH = "/content/drive/MyDrive/Projects/Tennis/v2_1/models/stage1_models/XGBoost_stage1_v2_1.pkl"
joblib.dump(xgbc, PATH)
print("Model Saved.")
print(elapsed_time)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model Saved.
45.79458832740784


In [22]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, brier_score_loss
import gc

val_preds = xgbc.predict(X_val)
val_prob = xgbc.predict_proba(X_val)[:, 1]

print(f"Accuracy: {accuracy_score(y_val, val_preds)}")
print(f"ROC-AUC: {roc_auc_score(y_val, val_prob)}")
print(f"Log Loss: {log_loss(y_val, val_prob)}")
print(f"Brier Score: {brier_score_loss(y_val, val_prob)}")

del xgbc
gc.collect()

Accuracy: 0.6629514726722586
ROC-AUC: 0.7271522895939464
Log Loss: 0.6072383424740485
Brier Score: 0.21047032032890403


131

## LightGBM 

### Base Model

In [10]:
from lightgbm import LGBMClassifier
import time


start_time = time.time()
lgbm = LGBMClassifier(objective="binary", metric="binary_logloss", boosting_type="gbdt", n_estimators=500, learning_rate=0.05, 
                      num_leaves=63, max_depth=-1, min_child_samples=20, subsample=0.8, colsample_bytree=0.8, reg_alpha=0, reg_lambda=0, 
                      random_state=42, verbose=-1, device= "gpu")

lgbm.fit(X_train, y_train)

elapsed_time = time.time() - start_time

MODEL_PATH = "/content/drive/MyDrive/Projects/Tennis/v2_1/models/base_models/lightGBM_base_v2_1.pkl"
joblib.dump(lgbm, MODEL_PATH)
print("Model Saved.")
print(elapsed_time)

Model Saved.
37.02686381340027


In [11]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, brier_score_loss
import gc

val_preds = lgbm.predict(X_val)
val_prob = lgbm.predict_proba(X_val)[:, 1]

print(f"Accuracy: {accuracy_score(y_val, val_preds)}")
print(f"ROC-AUC: {roc_auc_score(y_val, val_prob)}")
print(f"Log Loss: {log_loss(y_val, val_prob)}")
print(f"Brier Score: {brier_score_loss(y_val, val_prob)}")

del lgbm
gc.collect()

Accuracy: 0.6610364242215329
ROC-AUC: 0.7270199222798046
Log Loss: 0.6078944331307093
Brier Score: 0.21068666798478608


176

### Hyperparameter Search

In [15]:
!pip install optuna

In [16]:
import optuna
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.metrics import log_loss
import time
import warnings

warnings.filterwarnings("ignore")

start_time = time.time()

y_train = y_train.squeeze()

def objective(trial):
    params = {
        "objective": "binary",
        "metric": "binary_logloss",
        "verbosity": -1,
        "boosting_type": "gbdt",

        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 800, step=100),
        "num_leaves": trial.suggest_int("num_leaves", 31, 127),
        "max_depth": trial.suggest_int("max_depth", -1, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 20, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 20, log=True),
    }

    scores = []

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]

        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        model = LGBMClassifier(**params, random_state=42, n_jobs=1, device="gpu")

        model.fit(X_fold_train, y_fold_train)

        probs = model.predict_proba(X_fold_val)
        score = log_loss(y_fold_val, probs)

        scores.append(score)

        trial.report(sum(scores) / (fold + 1), fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return sum(scores) / len(scores)


study = optuna.create_study(
    direction="minimize",
    study_name="LightGBM Classifier",
    pruner=optuna.pruners.MedianPruner(),
)

study.optimize(objective, n_trials=30, show_progress_bar=True)

print("Elapsed Time:", time.time() - start_time)

[I 2026-11-21 12:12:25,978] A new study created in memory with name: LightGBM Classifier


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-11-21 12:14:40,044] Trial 0 finished with value: 0.5935595386442066 and parameters: {'learning_rate': 0.019235475787509917, 'n_estimators': 700, 'num_leaves': 101, 'max_depth': 9, 'min_child_samples': 42, 'subsample': 0.6273561186662037, 'colsample_bytree': 0.83776195329109, 'reg_alpha': 0.010175452700870002, 'reg_lambda': 2.74737018476127}. Best is trial 0 with value: 0.5935595386442066.
[I 2026-11-21 12:15:04,765] Trial 1 finished with value: 0.593981030155423 and parameters: {'learning_rate': 0.0192337416133564, 'n_estimators': 700, 'num_leaves': 55, 'max_depth': 2, 'min_child_samples': 17, 'subsample': 0.7254379296828996, 'colsample_bytree': 0.8595817699012153, 'reg_alpha': 7.019798602638217, 'reg_lambda': 5.518110618544593}. Best is trial 0 with value: 0.5935595386442066.
[I 2026-11-21 12:15:16,834] Trial 2 finished with value: 0.6107312063232732 and parameters: {'learning_rate': 0.01696647008060998, 'n_estimators': 200, 'num_leaves': 125, 'max_depth': 1, 'min_child_sample

In [17]:
study.best_value

0.5908770314101881

In [18]:
best_params = study.best_params
print("Best Parameters: ", best_params)

Best Parameters:  {'learning_rate': 0.033793944188534604, 'n_estimators': 200, 'num_leaves': 71, 'max_depth': 0, 'min_child_samples': 66, 'subsample': 0.8617440625590145, 'colsample_bytree': 0.6197654982663824, 'reg_alpha': 0.0428872014348539, 'reg_lambda': 3.364721457536489}


### Best Model

In [19]:
from lightgbm import LGBMClassifier
from google.colab import drive
import joblib
import time

start_time = time.time()

lgbmc = LGBMClassifier(**best_params, random_state=42, n_jobs=-1, device="gpu")

lgbmc.fit(X_train, y_train)

elapsed_time = time.time() - start_time

drive.mount('/content/drive')
PATH = "/content/drive/MyDrive/Projects/Tennis/v2_1/models/stage1_models/lightGBM_stage1_v2_1.pkl"
joblib.dump(lgbmc, PATH)
print("Model Saved.")
print(elapsed_time)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model Saved.
19.940781831741333


In [20]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, brier_score_loss

val_preds = lgbmc.predict(X_val)
val_prob = lgbmc.predict_proba(X_val)[:, 1]

print(f"Accuracy: {accuracy_score(y_val, val_preds)}")
print(f"ROC-AUC: {roc_auc_score(y_val, val_prob)}")
print(f"Log Loss: {log_loss(y_val, val_prob)}")
print(f"Brier Score: {brier_score_loss(y_val, val_prob)}")

Accuracy: 0.6623003561990118
ROC-AUC: 0.7264143392503581
Log Loss: 0.6079592810944285
Brier Score: 0.21075180090896053


## CatBoost

In [21]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 60.7 MB/s  0:00:016m0:00:0100:01


### Base Model

In [22]:
from catboost import CatBoostClassifier
import time

start_time = time.time()
cat = CatBoostClassifier(loss_function="Logloss", eval_metric="Logloss", random_seed=42, verbose=False, task_type="GPU", devices='0')

cat.fit(X_train, y_train)
elapsed_time = time.time() - start_time

MODEL_PATH = "/content/drive/MyDrive/Projects/Tennis/v2_1/models/base_models/catBoost_base_v2_1.pkl"
joblib.dump(cat, MODEL_PATH)
print("Model Saved.")
print(elapsed_time)

Model Saved.
14.562950849533081


In [23]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, brier_score_loss
import gc

val_preds = cat.predict(X_val)
val_prob = cat.predict_proba(X_val)[:, 1]

print(f"Accuracy: {accuracy_score(y_val, val_preds)}")
print(f"ROC-AUC: {roc_auc_score(y_val, val_prob)}")
print(f"Log Loss: {log_loss(y_val, val_prob)}")
print(f"Brier Score: {brier_score_loss(y_val, val_prob)}")

del cat
gc.collect()

Accuracy: 0.6623003561990118
ROC-AUC: 0.7268506599423816
Log Loss: 0.6074085386639825
Brier Score: 0.21057403080327378


0

### Hyperparameter Search

In [24]:
import optuna
from catboost import CatBoostClassifier
from sklearn.metrics import log_loss
import time

start_time = time.time()

def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 200, 800, step=100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.30, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 20),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 50),
    }

    scores = []

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]

        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        model = CatBoostClassifier(loss_function="Logloss", eval_metric="Logloss", random_seed=42, task_type="GPU", devices='0', 
        verbose=False, bootstrap_type="Bernoulli", **params)

        model.fit(X_fold_train, y_fold_train)

        probs = model.predict_proba(X_fold_val)
        score = log_loss(y_fold_val, probs)

        scores.append(score)

        trial.report(sum(scores) / (fold + 1), fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return sum(scores) / len(scores)


study = optuna.create_study(
    direction="minimize",
    study_name="CatBoost Classifier",
    pruner=optuna.pruners.MedianPruner(),
)

study.optimize(objective, n_trials=30, show_progress_bar=True)

print("Elapsed Time:", time.time() - start_time)

[I 2026-11-21 12:24:07,882] A new study created in memory with name: CatBoost Classifier


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-11-21 12:27:18,100] Trial 0 finished with value: 0.5923549829861076 and parameters: {'iterations': 700, 'learning_rate': 0.03957049703604509, 'depth': 9, 'l2_leaf_reg': 4.754345807912314, 'subsample': 0.6798426458218578, 'min_data_in_leaf': 12}. Best is trial 0 with value: 0.5923549829861076.
[I 2026-11-21 12:32:54,463] Trial 1 finished with value: 0.5908893401478353 and parameters: {'iterations': 700, 'learning_rate': 0.02853941328242948, 'depth': 10, 'l2_leaf_reg': 14.949430106280575, 'subsample': 0.8547204995682676, 'min_data_in_leaf': 15}. Best is trial 1 with value: 0.5908893401478353.
[I 2026-11-21 12:33:10,395] Trial 2 finished with value: 0.5903143825671755 and parameters: {'iterations': 400, 'learning_rate': 0.052589882953634384, 'depth': 6, 'l2_leaf_reg': 9.055116745886924, 'subsample': 0.9708099869042665, 'min_data_in_leaf': 15}. Best is trial 2 with value: 0.5903143825671755.
[I 2026-11-21 12:33:45,494] Trial 3 finished with value: 0.5911405585835134 and parameters:

In [25]:
best_params = study.best_params
print("Best Parameters: ", best_params)

Best Parameters:  {'iterations': 400, 'learning_rate': 0.033472977465455474, 'depth': 7, 'l2_leaf_reg': 10.480703717038644, 'subsample': 0.9550527559249474, 'min_data_in_leaf': 19}


In [26]:
study.best_value

0.590275330652979

### Best Model

In [27]:
from catboost import CatBoostClassifier
from google.colab import drive
import joblib
import time

start_time = time.time()

cat = CatBoostClassifier(loss_function="Logloss", eval_metric="Logloss", random_seed=42, task_type="GPU", devices='0', 
                         verbose=False, bootstrap_type="Bernoulli", **best_params)

cat.fit(X_train, y_train)

elapsed_time = time.time() - start_time

drive.mount('/content/drive')
PATH = "/content/drive/MyDrive/Projects/Tennis/v2_1/models/stage1_models/catBoost_stage1_v2_1.pkl"
joblib.dump(cat, PATH)
print("Model Saved.")
print(elapsed_time)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model Saved.
7.499335050582886


In [28]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, brier_score_loss
import gc

val_preds = cat.predict(X_val)
val_prob = cat.predict_proba(X_val)[:, 1]

print(f"Accuracy: {accuracy_score(y_val, val_preds)}")
print(f"ROC-AUC: {roc_auc_score(y_val, val_prob)}")
print(f"Log Loss: {log_loss(y_val, val_prob)}")
print(f"Brier Score: {brier_score_loss(y_val, val_prob)}")

del cat
gc.collect()

Accuracy: 0.6611896280975909
ROC-AUC: 0.7257860940213232
Log Loss: 0.6081620509870017
Brier Score: 0.21089959812673353


6654

# **More Fine Tuning**

In [21]:
!pip install optuna

## LightGBM

### Parameter Search

In [22]:
import time
import warnings
import numpy as np
import optuna
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.metrics import log_loss
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings("ignore")
start_time = time.time()

y_train = y_train.squeeze()

def objective(trial):

    learning_rate = trial.suggest_float("learning_rate", 0.003, 0.15, log=True)
    n_estimators = 1500

    max_depth = trial.suggest_categorical(
        "max_depth",
        [-1] + list(range(4, 17))
    )

    if max_depth == -1:
        num_leaves = trial.suggest_int("num_leaves", 31, 511)
    else:
        max_theoretical_leaves = min(511, 2 ** max_depth)
        num_leaves = trial.suggest_int(
            "num_leaves",
            16,
            max_theoretical_leaves
        )

    min_child_samples = trial.suggest_int("min_child_samples", 5, 150)
    min_split_gain = trial.suggest_float("min_split_gain", 0.0, 1.0)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    subsample_freq = trial.suggest_int("subsample_freq", 0, 10)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0)
    reg_alpha = trial.suggest_float("reg_alpha", 1e-5, 100, log=True)
    reg_lambda = trial.suggest_float("reg_lambda", 1e-5, 100, log=True)
    # max_bin = trial.suggest_categorical("max_bin", [127, 255, 511])
    extra_trees = trial.suggest_categorical("extra_trees", [False, True])

    scores = []

    tscv = DateGroupedCV(n_splits=3)

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):

        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]

        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        model = LGBMClassifier(objective="binary", metric="binary_logloss", boosting_type="gbdt", random_state=42, device="gpu", 
                                n_jobs=1, verbosity=-1, 
                                learning_rate=learning_rate, n_estimators=n_estimators, max_depth=max_depth, num_leaves=num_leaves, 
                                min_child_samples=min_child_samples, min_split_gain=min_split_gain, subsample=subsample, 
                                subsample_freq=subsample_freq, colsample_bytree=colsample_bytree, reg_alpha=reg_alpha, 
                                reg_lambda=reg_lambda, extra_trees=extra_trees)

        model.fit(X_fold_train, y_fold_train)

        probs = model.predict_proba(X_fold_val)
        score = log_loss(y_fold_val, probs)

        scores.append(score)

        trial.report(sum(scores) / (fold + 1), fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return sum(scores) / len(scores)


study = optuna.create_study(direction="minimize", study_name="LightGBM_TSCV", pruner=optuna.pruners.MedianPruner(),)

study.optimize(objective, n_trials=60, show_progress_bar=True)

print("\n" + "=" * 40)
print(f"Best Parameters: {study.best_params}")
print(f"Best Cross-Validation Loss: {study.best_value:.5f}")
print(f"Total Tuning Time: {time.time() - start_time:.2f} seconds")
print("=" * 40)

[I 2026-11-22 02:14:12,345] A new study created in memory with name: LightGBM_TSCV


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-11-22 02:15:24,815] Trial 0 finished with value: 0.5894347486709904 and parameters: {'learning_rate': 0.03520621477890112, 'max_depth': 12, 'num_leaves': 119, 'min_child_samples': 44, 'min_split_gain': 0.5952966064337283, 'subsample': 0.8818862152442108, 'subsample_freq': 5, 'colsample_bytree': 0.9185711264173115, 'reg_alpha': 0.05800832421921617, 'reg_lambda': 0.7948151822641863, 'extra_trees': True}. Best is trial 0 with value: 0.5894347486709904.
[I 2026-11-22 02:16:17,423] Trial 1 finished with value: 0.6162516465352171 and parameters: {'learning_rate': 0.14436622674143584, 'max_depth': 6, 'num_leaves': 56, 'min_child_samples': 55, 'min_split_gain': 0.9613682038977993, 'subsample': 0.9398345010554648, 'subsample_freq': 5, 'colsample_bytree': 0.6168780934801965, 'reg_alpha': 0.004696899920067893, 'reg_lambda': 3.4889388761046214e-05, 'extra_trees': False}. Best is trial 0 with value: 0.5894347486709904.
[I 2026-11-22 02:17:42,526] Trial 2 finished with value: 0.6154552933754

In [23]:
study.best_value

0.5881516826124186

In [24]:
best_params = study.best_params
print("Best Parameters: ", best_params)

Best Parameters:  {'learning_rate': 0.013466911951241573, 'max_depth': 12, 'num_leaves': 254, 'min_child_samples': 111, 'min_split_gain': 0.21664128093515195, 'subsample': 0.9695856589420997, 'subsample_freq': 10, 'colsample_bytree': 0.9338461695042399, 'reg_alpha': 0.25189656779247516, 'reg_lambda': 1.3798004568062485, 'extra_trees': True}


### Best Model

In [25]:
from lightgbm import LGBMClassifier
import time
import joblib
from google.colab import drive

start_time = time.time()

lgbmc = LGBMClassifier(**best_params, n_estimators=1500, objective="binary", random_state=42, n_jobs=-1, device="gpu")

lgbmc.fit(X_train, y_train)
elapsed_time = time.time() - start_time

drive.mount('/content/drive')
PATH = "/content/drive/MyDrive/Projects/Tennis/v2_1/models/stage2_models/lightGBM_stage2_v2_1.pkl"
joblib.dump(lgbmc, PATH)
print("Model Saved.")
print(elapsed_time)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model Saved.
78.09116864204407


In [26]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, brier_score_loss
import gc

val_preds = lgbmc.predict(X_val)
val_prob = lgbmc.predict_proba(X_val)[:, 1]

print(f"Accuracy: {accuracy_score(y_val, val_preds)}")
print(f"ROC-AUC: {roc_auc_score(y_val, val_prob)}")
print(f"Log Loss: {log_loss(y_val, val_prob)}")
print(f"Brier Score: {brier_score_loss(y_val, val_prob)}")

del lgbmc
gc.collect()

Accuracy: 0.6635642881764908
ROC-AUC: 0.728120376862873
Log Loss: 0.6065498156584063
Brier Score: 0.21018749498265757


439

## CatBoost

In [27]:
!pip install catboost

### Parameter Search

In [28]:
import time
import warnings
import optuna
from catboost import CatBoostClassifier
from sklearn.metrics import log_loss
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings("ignore")
start_time = time.time()

y_train = y_train.squeeze()

def objective(trial):

    learning_rate = trial.suggest_float("learning_rate", 0.003, 0.15, log=True)
    depth = trial.suggest_int("depth", 4, 10)
    min_data_in_leaf = trial.suggest_int("min_data_in_leaf", 1, 100)
    l2_leaf_reg = trial.suggest_float("l2_leaf_reg", 1, 50)
    random_strength = trial.suggest_float("random_strength", 1e-3, 20, log=True)

    bootstrap_type = trial.suggest_categorical(
        "bootstrap_type",
        ["Bernoulli", "MVS", "Bayesian"]
    )

    params = {}

    if bootstrap_type == "Bernoulli":
        params["subsample"] = trial.suggest_float("subsample", 0.5, 1.0)

    elif bootstrap_type == "Bayesian":
        params["bagging_temperature"] = trial.suggest_float(
            "bagging_temperature",
            0,
            10
        )

    # border_count = trial.suggest_categorical("border_count", [64, 128, 254])

    # rsm = trial.suggest_float("rsm", 0.5, 1.0)

    scores = []

    tscv = DateGroupedCV(n_splits=3)

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):

        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]

        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        model = CatBoostClassifier(loss_function="Logloss", eval_metric="Logloss", random_seed=42, task_type="GPU", devices='0', verbose=False,
                                    iterations=1500, learning_rate=learning_rate, depth=depth, min_data_in_leaf=min_data_in_leaf, 
                                    l2_leaf_reg=l2_leaf_reg, random_strength=random_strength, bootstrap_type=bootstrap_type, 
                                    **params)

        model.fit(X_fold_train, y_fold_train)

        probs = model.predict_proba(X_fold_val)
        score = log_loss(y_fold_val, probs)

        scores.append(score)

        trial.report(sum(scores) / (fold + 1), fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return sum(scores) / len(scores)


study = optuna.create_study(direction="minimize", study_name="CatBoost_TSCV", pruner=optuna.pruners.MedianPruner(),)

study.optimize(objective, n_trials=60, show_progress_bar=True)

print("\n" + "=" * 40)
print(f"Best Parameters: {study.best_params}")
print(f"Best Cross-Validation Loss: {study.best_value:.5f}")
print(f"Total Tuning Time: {time.time() - start_time:.2f} seconds")
print("=" * 40)

[I 2026-11-22 03:50:03,381] A new study created in memory with name: CatBoost_TSCV


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-11-22 03:51:15,099] Trial 0 finished with value: 0.6307409270580074 and parameters: {'learning_rate': 0.0031927284566025975, 'depth': 5, 'min_data_in_leaf': 86, 'l2_leaf_reg': 8.897790159599744, 'random_strength': 0.2383370539268867, 'bootstrap_type': 'MVS'}. Best is trial 0 with value: 0.6307409270580074.
[I 2026-11-22 03:52:21,516] Trial 1 finished with value: 0.5910151299172797 and parameters: {'learning_rate': 0.017529571618398243, 'depth': 4, 'min_data_in_leaf': 5, 'l2_leaf_reg': 5.977244138672617, 'random_strength': 0.03487989833868037, 'bootstrap_type': 'Bernoulli', 'subsample': 0.9178495179430546}. Best is trial 1 with value: 0.5910151299172797.
[I 2026-11-22 03:54:03,436] Trial 2 finished with value: 0.594505468746173 and parameters: {'learning_rate': 0.029012935538942766, 'depth': 6, 'min_data_in_leaf': 73, 'l2_leaf_reg': 33.72498359614314, 'random_strength': 0.003994788971185075, 'bootstrap_type': 'Bayesian', 'bagging_temperature': 8.308413082186082}. Best is trial 1

In [29]:
best_params = study.best_params
print("Best Parameters: ", best_params)

Best Parameters:  {'learning_rate': 0.014114766673936081, 'depth': 8, 'min_data_in_leaf': 50, 'l2_leaf_reg': 16.410186810629597, 'random_strength': 0.0017896171107572194, 'bootstrap_type': 'Bernoulli', 'subsample': 0.7155459584170225}


In [30]:
study.best_value

0.5896536176415541

### Best Model

In [31]:
from catboost import CatBoostClassifier
from google.colab import drive
import joblib
import time

start_time = time.time()

cat = CatBoostClassifier(**best_params, iterations=1500, loss_function="Logloss", eval_metric="Logloss", random_seed=42, task_type="GPU", verbose=False)

cat.fit(X_train, y_train)

elapsed_time = time.time() - start_time

drive.mount('/content/drive')
PATH = "/content/drive/MyDrive/Projects/Tennis/v2_1/models/stage2_models/catBoost_stage2_v2_1.pkl"
joblib.dump(cat, PATH)
print("Model Saved.")
print(elapsed_time)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model Saved.
28.947725296020508


In [32]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, brier_score_loss

val_preds = cat.predict(X_val)
val_prob = cat.predict_proba(X_val)[:, 1]

print(f"Accuracy: {accuracy_score(y_val, val_preds)}")
print(f"ROC-AUC: {roc_auc_score(y_val, val_prob)}")
print(f"Log Loss: {log_loss(y_val, val_prob)}")
print(f"Brier Score: {brier_score_loss(y_val, val_prob)}")

Accuracy: 0.6624152591060554
ROC-AUC: 0.7274110065925877
Log Loss: 0.6070954725230803
Brier Score: 0.21042251769976741


## XGBoost

### Search

In [33]:
import time
import warnings
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import log_loss

warnings.filterwarnings("ignore")
start_time = time.time()

y_train = y_train.squeeze()

def objective(trial):

    learning_rate = trial.suggest_float("learning_rate", 0.003, 0.15, log=True)
    max_depth = trial.suggest_int("max_depth", 3, 15)
    min_child_weight = trial.suggest_float("min_child_weight", 1, 20)
    gamma = trial.suggest_float("gamma", 0, 10)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0)
    colsample_bylevel = trial.suggest_float("colsample_bylevel", 0.5, 1.0)
    reg_alpha = trial.suggest_float("reg_alpha", 1e-5, 100, log=True)
    reg_lambda = trial.suggest_float("reg_lambda", 1e-5, 100, log=True)
    max_bin = trial.suggest_categorical("max_bin", [256, 512, 1024])
    grow_policy = trial.suggest_categorical(
        "grow_policy",
        ["depthwise", "lossguide"]
    )

    scores = []

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):

        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]

        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        model = XGBClassifier(objective="binary:logistic", eval_metric="logloss", tree_method="hist", device="cuda",
                            random_state=42, n_jobs=1, n_estimators=1500, 
                            learning_rate=learning_rate, max_depth=max_depth, min_child_weight=min_child_weight, gamma=gamma, 
                            subsample=subsample, colsample_bytree=colsample_bytree, colsample_bylevel=colsample_bylevel, 
                            reg_alpha=reg_alpha, reg_lambda=reg_lambda, max_bin=max_bin, grow_policy=grow_policy, 
                            )

        model.fit(X_fold_train, y_fold_train, verbose=False)

        probs = model.predict_proba(X_fold_val)
        score = log_loss(y_fold_val, probs)

        scores.append(score)

        trial.report(sum(scores) / (fold + 1), fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return sum(scores) / len(scores)


study = optuna.create_study(direction="minimize", study_name="XGBoost_TSCV", pruner=optuna.pruners.MedianPruner(),)

study.optimize(objective, n_trials=60, show_progress_bar=True)

print("\n" + "=" * 40)
print(f"Best Parameters: {study.best_params}")
print(f"Best Cross-Validation Loss: {study.best_value:.5f}")
print(f"Total Tuning Time: {time.time() - start_time:.2f} seconds")
print("=" * 40)

[I 2026-11-22 07:57:10,226] A new study created in memory with name: XGBoost_TSCV


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-11-22 08:00:39,669] Trial 0 finished with value: 0.5928995882843116 and parameters: {'learning_rate': 0.008941246328517043, 'max_depth': 10, 'min_child_weight': 4.867215167683259, 'gamma': 0.7096912780452336, 'subsample': 0.6931471970673702, 'colsample_bytree': 0.7747176842325842, 'colsample_bylevel': 0.9490056380155414, 'reg_alpha': 19.21688790735983, 'reg_lambda': 3.6865255100442975, 'max_bin': 1024, 'grow_policy': 'lossguide'}. Best is trial 0 with value: 0.5928995882843116.
[I 2026-11-22 08:01:55,346] Trial 1 finished with value: 0.5992407394210223 and parameters: {'learning_rate': 0.032387099747203135, 'max_depth': 10, 'min_child_weight': 1.9377668791947735, 'gamma': 1.9707990091300898, 'subsample': 0.8314377461757227, 'colsample_bytree': 0.5168223257215404, 'colsample_bylevel': 0.7064416340352122, 'reg_alpha': 0.10869221802468437, 'reg_lambda': 0.9145653783298234, 'max_bin': 256, 'grow_policy': 'lossguide'}. Best is trial 0 with value: 0.5928995882843116.
[I 2026-11-22 08

In [34]:
best_params= study.best_params
print("Best Parameters: ", best_params)

Best Parameters:  {'learning_rate': 0.007230441910681208, 'max_depth': 6, 'min_child_weight': 18.550607004196195, 'gamma': 1.0494996131461967, 'subsample': 0.7272769085004807, 'colsample_bytree': 0.5370259891345331, 'colsample_bylevel': 0.5211341466587978, 'reg_alpha': 4.943281159494726e-05, 'reg_lambda': 2.15801421514572, 'max_bin': 256, 'grow_policy': 'depthwise'}


In [35]:
study.best_value

0.5886869218171273

### Best Model

In [36]:
from xgboost import XGBClassifier
import joblib
from google.colab import drive
import time

start_time = time.time()

xgbc = XGBClassifier(objective="binary:logistic", eval_metric="logloss", tree_method="hist", device="cuda", random_state=42, 
                  n_estimators=1500, **study.best_params)

xgbc.fit(X_train, y_train)
elapsed_time = time.time() - start_time

#saving the models
drive.mount('/content/drive')
PATH = "/content/drive/MyDrive/Projects/Tennis/v2_1/models/stage2_models/XGBoost_stage2_v2_1.pkl"
joblib.dump(xgbc, PATH)
print("Model Saved.")
print(elapsed_time)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model Saved.
20.108625650405884


In [37]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, brier_score_loss
import gc

val_preds = xgbc.predict(X_val)
val_prob = xgbc.predict_proba(X_val)[:, 1]

print(f"Accuracy: {accuracy_score(y_val, val_preds)}")
print(f"ROC-AUC: {roc_auc_score(y_val, val_prob)}")
print(f"Log Loss: {log_loss(y_val, val_prob)}")
print(f"Brier Score: {brier_score_loss(y_val, val_prob)}")

del xgbc
gc.collect()

Accuracy: 0.6621088513539393
ROC-AUC: 0.7276792502823299
Log Loss: 0.6067864621362556
Brier Score: 0.21028100139094058


6782

# **Validation Predictions**

In [38]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Logistic Regression

In [39]:
import joblib
from sklearn.linear_model import LogisticRegression
import numpy as np
import gc

PATH = "/content/drive/MyDrive/Projects/Tennis/v2_1/models/stage1_models/logReg_stage1_v2_1.pkl"
logReg = joblib.load(PATH)

logreg_preds = logReg.predict_proba(X_val_log)
logreg_preds = np.clip(logreg_preds, 0, 1)

del logReg
gc.collect()

0

## SGD Classifier

In [40]:
import joblib
from sklearn.linear_model import SGDClassifier
import numpy as np

PATH = "/content/drive/MyDrive/Projects/Tennis/v2_1/models/stage1_models/sgd_stage1_v2_1.pkl"
sgd = joblib.load(PATH)

sgd_preds = np.clip(sgd.predict_proba(X_val_log), 0, 1)

del sgd
gc.collect()

0

## Random Forests

In [41]:
from sklearn.ensemble import RandomForestClassifier
import joblib

PATH = "/content/drive/MyDrive/Projects/Tennis/v2_1/models/stage1_models/randomForest_stage1_v2_1.pkl"

rf = joblib.load(PATH)

rf_preds = rf.predict_proba(X_val)
rf_preds = np.clip(rf_preds, 0, 1)

del rf
gc.collect()

24

## XGBoost

In [43]:
from xgboost import XGBClassifier
import joblib

PATH = "/content/drive/MyDrive/Projects/Tennis/v2_1/models/stage2_models/XGBoost_stage2_v2_1.pkl"

xgb = joblib.load(PATH)

xgb_preds = np.clip(xgb.predict_proba(X_val), 0, 1)

del xgb
gc.collect()

41

## CatBoost

In [44]:
!pip install catboost

In [45]:
from catboost import CatBoostClassifier
import joblib

PATH = "/content/drive/MyDrive/Projects/Tennis/v2_1/models/stage2_models/catBoost_stage2_v2_1.pkl"

cat = joblib.load(PATH)

cat_preds = np.clip(cat.predict_proba(X_val), 0, 1)

del cat
gc.collect()

30

## LightGBM

In [46]:
from lightgbm import LGBMClassifier
import joblib

PATH = "/content/drive/MyDrive/Projects/Tennis/v2_1/models/stage2_models/lightGBM_stage2_v2_1.pkl"

lgbm = joblib.load(PATH)

lgbm_preds = np.clip(lgbm.predict_proba(X_val), 0, 1)

del lgbm
gc.collect()

4

## Saving Predictions

In [47]:
print("y_val:", y_val.shape)
print("logreg:", logreg_preds.shape)
print("sgd:", sgd_preds.shape)
print("rf:", rf_preds.shape)
print("lgbm:", lgbm_preds.shape)
print("xgb:", xgb_preds.shape)
print("cat:", cat_preds.shape)

y_val: (26109,)
logreg: (26109, 2)
sgd: (26109, 2)
rf: (26109, 2)
lgbm: (26109, 2)
xgb: (26109, 2)
cat: (26109, 2)


In [54]:
import pandas as pd
import numpy as np

val_ids = metadata_by_split["val"]
pred_df = pd.DataFrame({
    "dataset_row": val_ids["dataset_row"].to_numpy(),
    "match_id": val_ids["match_id"].to_numpy() if "match_id" in val_ids else None,
    "playerA_wins": y_val.to_numpy(),
    "logreg": logreg_preds[:, 1],
    "sgd": sgd_preds[:, 1],
    "random_forest": rf_preds[:, 1],
    "lightgbm": lgbm_preds[:, 1],
    "xgboost": xgb_preds[:, 1],
    "catboost": cat_preds[:, 1],
})
assert pred_df["dataset_row"].is_unique
assert np.isfinite(pred_df[["logreg", "sgd", "random_forest",
                            "lightgbm", "xgboost", "catboost"]].to_numpy()).all()
path = PROJECT_DIR / "predictions/v2_1_validation_individual.parquet"
pred_df.to_parquet(path, index=False)
print("Saved:", path)


Saved: /content/drive/MyDrive/Projects/Tennis/v2_1/predictions/v2_1_validation_individual.parquet


## Evaluating Log Loss

In [55]:
from sklearn.metrics import log_loss

print("Logistic Regression: ", log_loss(y_true= y_val, y_pred= logreg_preds[:, 1]))
print("SGD: ", log_loss(y_true= y_val, y_pred= sgd_preds[:, 1]))
print("Random Forest: ", log_loss(y_true= y_val, y_pred= rf_preds[:, 1]))
print("LightGBM: ", log_loss(y_true= y_val, y_pred= lgbm_preds[:, 1]))
print("XGBoost: ", log_loss(y_true= y_val, y_pred= xgb_preds[:, 1]))
print("CatBoost: ", log_loss(y_true= y_val, y_pred= cat_preds[:, 1]))

Logistic Regression:  0.6088086433713008
SGD:  0.6089423428516051
Random Forest:  0.6105969234947792
LightGBM:  0.6065498156584063
XGBoost:  0.6067864621362556
CatBoost:  0.6070954725230803


## Correlation

In [56]:
pred_df[["logreg","sgd", "random_forest", "xgboost","catboost","lightgbm"]].corr()

,logreg,sgd,random_forest,xgboost,catboost,lightgbm
logreg,1.000000,0.997849,0.962036,0.979726,0.980343,0.976773
sgd,0.997849,1.000000,0.966796,0.980848,0.981008,0.976244
random_forest,0.962036,0.966796,1.000000,0.982012,0.978661,0.971484
xgboost,0.979726,0.980848,0.982012,1.000000,0.996039,0.992294
catboost,0.980343,0.981008,0.978661,0.996039,1.000000,0.990698
lightgbm,0.976773,0.976244,0.971484,0.992294,0.990698,1.000000


In [57]:
#for error

y_val = np.ravel(y_val)
error_df = pd.DataFrame({
    "logreg": y_val - logreg_preds[:, 1],
    "sgd": y_val - sgd_preds[:, 1],
    "random_forest": y_val - rf_preds[:, 1], 
    "lgbm": y_val - lgbm_preds[:, 1],
    "xgb": y_val - xgb_preds[:, 1],
    "cat": y_val - cat_preds[:, 1]
})

error_df.corr()

,logreg,sgd,random_forest,lgbm,xgb,cat
logreg,1.000000,0.999542,0.992448,0.995107,0.995872,0.995970
sgd,0.999542,1.000000,0.993601,0.994973,0.996135,0.996121
random_forest,0.992448,0.993601,1.000000,0.993879,0.996307,0.995574
lgbm,0.995107,0.994973,0.993879,1.000000,0.998338,0.998018
xgb,0.995872,0.996135,0.996307,0.998338,1.000000,0.999183
cat,0.995970,0.996121,0.995574,0.998018,0.999183,1.000000


# **Ensemble**

In [58]:
# Reload validation probabilities with original V1.1 row identities.
import pandas as pd

PATH = PROJECT_DIR / "predictions/v2_1_validation_individual.parquet"
pred_df = pd.read_parquet(PATH)
if not pred_df["dataset_row"].equals(metadata_by_split["val"]["dataset_row"]):
    raise ValueError("Validation prediction order does not match V1.1 metadata")
y_val = pred_df["playerA_wins"]
logreg_preds = pred_df["logreg"]
sgd_preds = pred_df["sgd"]
lightgbm_preds = pred_df["lightgbm"]
xgboost_preds = pred_df["xgboost"]
catboost_preds = pred_df["catboost"]
rf_preds = pred_df["random_forest"]


## Cat + LGBM + XGB

### Equal Ensemble

In [59]:
from sklearn.metrics import log_loss

ensemble_equal = (lightgbm_preds + xgboost_preds + catboost_preds)/3

print("Equal Ensemble (Ex. Ridge): ", log_loss(y_val, ensemble_equal))

Equal Ensemble (Ex. Ridge):  0.6062779243841911


### Inverse MAE Weighted Ensemble

In [60]:

ll_lgbm = log_loss(y_val, lightgbm_preds)
ll_xgb  = log_loss(y_val, xgboost_preds)
ll_cb   = log_loss(y_val, catboost_preds)

w_lgbm = 1 / ll_lgbm
w_xgb  = 1 / ll_xgb
w_cb   = 1 / ll_cb

total = w_lgbm + w_xgb + w_cb

w_lgbm /= total
w_xgb  /= total
w_cb   /= total

ensemble = (
    w_lgbm * lightgbm_preds +
    w_xgb  * xgboost_preds +
    w_cb   * catboost_preds
)

print(log_loss(y_val, ensemble))

0.606277805954551


### Optimized Ensemble

In [61]:
import numpy as np
from scipy.optimize import minimize
from sklearn.metrics import log_loss

predictions = np.vstack([lightgbm_preds, xgboost_preds, catboost_preds])

def objective(weights):
    ensemble = np.average(predictions, axis=0, weights=weights)
    return log_loss(y_val, ensemble)

constraints = ({
    "type": "eq",
    "fun": lambda w: np.sum(w) - 1
})

bounds = [(0,1)] * 3

initial = np.array([1/3,1/3,1/3])

result = minimize(objective, initial, bounds=bounds, constraints=constraints)
print(result)
if not result.success:
    raise RuntimeError("Validation ensemble optimization did not converge")

     message: Optimization terminated successfully
     success: True
      status: 0
         fun: 0.6062779243841911
           x: [ 3.333e-01  3.333e-01  3.333e-01]
         nit: 1
         jac: [-4.863e-04  1.608e-04  3.255e-04]
        nfev: 4
        njev: 1
 multipliers: [ 9.035e-21]


## Cat + LGBM + XGB + LogReg

### Equal Weighted

In [62]:
from sklearn.metrics import log_loss

ensemble_equal = (lightgbm_preds + xgboost_preds + catboost_preds + logreg_preds)/4

print("Equal Ensemble (Inc. Ridge): ", log_loss(y_val, ensemble_equal))

Equal Ensemble (Inc. Ridge):  0.6061537712380646


## Cat + LGBM + XGB + SGD

### Equal Weighted

In [64]:
from sklearn.metrics import log_loss

ensemble_equal = (lightgbm_preds + xgboost_preds + catboost_preds + sgd_preds)/4

print("Equal Ensemble (Inc. Enet): ", log_loss(y_val, ensemble_equal))

Equal Ensemble (Inc. Enet):  0.6062113897459555


## Cat + LGBM + XGB + RF

### Equal Ensemble

In [66]:
from sklearn.metrics import log_loss

ensemble_equal2 = (lightgbm_preds + xgboost_preds + catboost_preds + rf_preds)/4

print("Equal Ensemble : ", log_loss(y_val, ensemble_equal2))

Equal Ensemble :  0.606549359595623


### Equal Ensemble

In [71]:
from sklearn.metrics import log_loss

ensemble_equal2 = (lightgbm_preds + xgboost_preds + catboost_preds + rf_preds)/4

print("Equal Ensemble : ", log_loss(y_val, ensemble_equal2))

Equal Ensemble :  0.606549359595623


## Cat + LGBM + XGB + RF + LogReg

### Equal Weighted

In [72]:
from sklearn.metrics import log_loss

ensemble_equal = (lightgbm_preds + rf_preds + catboost_preds + logreg_preds + xgboost_preds)/5

print("Equal Ensemble (Inc. Ridge): ", log_loss(y_val, ensemble_equal))

Equal Ensemble (Inc. Ridge):  0.6063299392875678


## Stacking

In [82]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import log_loss, roc_auc_score
import pandas as pd
import numpy as np
import joblib
import gc

# Tuned base models. SGD retains its fitted Pipeline structure and log features.
MODEL_DIR = PROJECT_DIR / "models"
stack_cols = ["xgboost", "lightgbm", "catboost", "sgd"]
templates = {
    "xgboost": joblib.load(MODEL_DIR / "stage2_models/XGBoost_stage2_v2_1.pkl"),
    "lightgbm": joblib.load(MODEL_DIR / "stage2_models/lightGBM_stage2_v2_1.pkl"),
    "catboost": joblib.load(MODEL_DIR / "stage2_models/catBoost_stage2_v2_1.pkl"),
    "sgd": joblib.load(MODEL_DIR / "stage1_models/sgd_stage1_v2_1.pkl"),
}

# Expanding, tournament-date-grouped out-of-fold predictions on training only.
n = len(X_train)
oof = {name: np.full(n, np.nan) for name in stack_cols}
oof_seen = np.zeros(n, dtype=bool)
tscv_stack = DateGroupedCV(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(tscv_stack.split(X_train), start=1):
    print(f"\nTraining stacking fold {fold}/5")
    x_tr, x_va = X_train.iloc[train_idx], X_train.iloc[val_idx]
    x_tr_log, x_va_log = X_train_log.iloc[train_idx], X_train_log.iloc[val_idx]
    y_tr = y_train.iloc[train_idx]

    for name in stack_cols:
        model = clone(templates[name])
        if name == "sgd":
            model.fit(x_tr_log, y_tr)
            probs = model.predict_proba(x_va_log)[:, 1]
        else:
            model.fit(x_tr, y_tr)
            probs = model.predict_proba(x_va)[:, 1]
        oof[name][val_idx] = np.asarray(probs).ravel()
        del model
        gc.collect()
    oof_seen[val_idx] = True
    print(f"Fold {fold} completed.")

del templates
X_stack_train = pd.DataFrame({name: oof[name][oof_seen] for name in stack_cols})
y_stack_train = y_train.iloc[oof_seen].reset_index(drop=True)
if not np.isfinite(X_stack_train.to_numpy()).all():
    raise ValueError("Stacking OOF probabilities are missing or non-finite")

# Tune meta-model on OOF predictions, keeping tournament dates together.
oof_dates = train_dates.iloc[oof_seen].reset_index(drop=True)
unique_dates = np.sort(oof_dates.unique())
meta_cv = [
    (
        np.flatnonzero(oof_dates.isin(unique_dates[tr]).to_numpy()),
        np.flatnonzero(oof_dates.isin(unique_dates[va]).to_numpy()),
    )
    for tr, va in TimeSeriesSplit(n_splits=3).split(unique_dates)
]
meta_search = GridSearchCV(
    LogisticRegression(max_iter=2000, random_state=42),
    param_grid={"C": np.logspace(-3, 3, 13)},
    scoring="neg_log_loss", cv=meta_cv, n_jobs=-1, refit=True,
)
meta_search.fit(X_stack_train, y_stack_train)
meta_model = meta_search.best_estimator_
print("\nBest meta-model C:", meta_search.best_params_["C"])
print("Meta CV log loss:", -meta_search.best_score_)

# Only validation data are evaluated here; test is not yet loaded.
if not pred_df["dataset_row"].equals(metadata_by_split["val"]["dataset_row"]):
    raise ValueError("Stacking validation rows do not match V1.1 metadata")
X_stack_val = pred_df[stack_cols].reset_index(drop=True)
stack_val_probs = meta_model.predict_proba(X_stack_val)[:, 1]
pred_df["stacked_probA"] = stack_val_probs
print("\nStacking validation log loss:", log_loss(y_val, stack_val_probs))
print("Stacking validation ROC-AUC:", roc_auc_score(y_val, stack_val_probs))
print(pd.DataFrame({"Model": stack_cols, "Coefficient": meta_model.coef_[0]}))
print("Intercept:", meta_model.intercept_[0])

ensemble_dir = MODEL_DIR / "ensembles"
ensemble_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(meta_model, ensemble_dir / "stack_xgb_lgbm_cat_sgd_v2_1.pkl")
pred_df.to_parquet(PROJECT_DIR / "predictions/v2_1_validation_individual.parquet", index=False)
print("Saved stacking meta-model and aligned validation probabilities.")



Training stacking fold 1/5
Fold 1 completed.

Training stacking fold 2/5
Fold 2 completed.

Training stacking fold 3/5
Fold 3 completed.

Training stacking fold 4/5
Fold 4 completed.

Training stacking fold 5/5
Fold 5 completed.

Best meta-model C: 3.1622776601683795
Meta CV log loss: 0.5860285709199156

Stacking validation log loss: 0.607709660340589
Stacking validation ROC-AUC: 0.7283523269988985
      Model  Coefficient
0   xgboost     1.255881
1  lightgbm     2.189187
2  catboost    -0.112027
3       sgd     1.485642
Intercept: -2.413410385864019
Saved stacking meta-model and aligned validation probabilities.


# **Retraining on the Training + Validation Set**

In [83]:
# Build exactly the V1.1 training + validation partitions; never include test.
X_train_val = pd.concat([X_train, X_val], ignore_index=True)
X_train_val_log = pd.concat([X_train_log, X_val_log], ignore_index=True)
y_train_val = pd.concat([y_train, y_val], ignore_index=True)
assert len(X_train_val) == len(X_train_val_log) == len(y_train_val)
assert X_train_val.columns.equals(X_train.columns)
assert X_train_val_log.columns.equals(X_train_log.columns)
print("Final retraining matches:", len(y_train_val))

Final retraining matches: 207741


## CatBoost

In [84]:
!pip install catboost

In [85]:
import joblib
import time
from sklearn.base import clone

start_time = time.time()
cat = clone(joblib.load(PROJECT_DIR / "models/stage2_models/catBoost_stage2_v2_1.pkl"))
cat.fit(X_train_val, y_train_val)
SAVE_PATH = PROJECT_DIR / "models/final_models/catboost_final_v2_1.pkl"
SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(cat, SAVE_PATH)
print("CatBoost final model saved. Training time:", time.time() - start_time)


CatBoost final model saved. Training time: 29.942777156829834


## LightGBM

In [86]:
import joblib
import time
from sklearn.base import clone

start_time = time.time()
lgbm = clone(joblib.load(PROJECT_DIR / "models/stage2_models/lightGBM_stage2_v2_1.pkl"))
lgbm.fit(X_train_val, y_train_val)
SAVE_PATH = PROJECT_DIR / "models/final_models/lightgbm_final_v2_1.pkl"
SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(lgbm, SAVE_PATH)
print("LightGBM final model saved. Training time:", time.time() - start_time)


LightGBM final model saved. Training time: 92.70297312736511


## XGBoost

In [87]:
import joblib
import time
from sklearn.base import clone

start_time = time.time()
xgb = clone(joblib.load(PROJECT_DIR / "models/stage2_models/XGBoost_stage2_v2_1.pkl"))
xgb.fit(X_train_val, y_train_val)
SAVE_PATH = PROJECT_DIR / "models/final_models/xgboost_final_v2_1.pkl"
SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(xgb, SAVE_PATH)
print("XGBoost final model saved. Training time:", time.time() - start_time)


XGBoost final model saved. Training time: 23.363287687301636


## SGD Classifier

In [88]:
import joblib
import time
from sklearn.base import clone

start_time = time.time()
sgd_final = clone(joblib.load(PROJECT_DIR / "models/stage1_models/sgd_stage1_v2_1.pkl"))
# Fit the scaler and classifier together on train + validation log features.
sgd_final.fit(X_train_val_log, y_train_val)
SAVE_PATH = PROJECT_DIR / "models/final_models/sgd_final_v2_1.pkl"
SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(sgd_final, SAVE_PATH)
print("SGD final model saved. Training time:", time.time() - start_time)

SGD final model saved. Training time: 4.800240993499756


# **Final Test Probabilities (V2.1)**

In [89]:
# Select the equal-weight four-model ensemble using validation predictions only.
from sklearn.metrics import log_loss
import numpy as np

validation_columns = ["catboost", "lightgbm", "xgboost", "sgd"]
ensemble_weights = np.full(len(validation_columns), 1.0 / len(validation_columns))
pred_df["match_win_probA"] = pred_df[validation_columns].to_numpy() @ ensemble_weights
print("Primary V2.1 ensemble:", dict(zip(validation_columns, ensemble_weights)))
print("Validation log loss:", log_loss(y_val, pred_df["match_win_probA"]))
pred_df.to_parquet(PROJECT_DIR / "predictions/v2_1_validation_individual.parquet", index=False)


Primary V2.1 ensemble: {'catboost': np.float64(0.25), 'lightgbm': np.float64(0.25), 'xgboost': np.float64(0.25), 'sgd': np.float64(0.25)}
Validation log loss: 0.6062113897459555


In [93]:

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    brier_score_loss, log_loss
)

BASE = Path("/content/drive/MyDrive/Projects/Tennis")
V2 = BASE / "v2_1"

# Load test predictions and actual match winners
preds = pd.read_parquet(
    V2 / "predictions/v2_1_test_match_probabilities.parquet"
)

metadata = pd.read_csv(
    BASE / "v1_1/datasets/matches_v1_1_metadata.csv"
)

test_meta = metadata.loc[
    metadata["split"] == "test",
    ["dataset_row", "playerA_wins"]
].rename(columns={"playerA_wins": "actual_winner"})

# Align by the original V1.1 match identifier
assert preds["dataset_row"].is_unique
assert test_meta["dataset_row"].is_unique

df = preds.merge(
    test_meta,
    on="dataset_row",
    how="inner",
    validate="one_to_one",
    sort=False
)

assert len(df) == len(preds) == len(test_meta), "Match alignment failed"

if "playerA_wins" in df.columns:
    assert np.array_equal(
        df["playerA_wins"].to_numpy(),
        df["actual_winner"].to_numpy()
    ), "Stored winners do not match metadata"

y_true = df["actual_winner"].astype(int)

# Evaluate primary ensemble, stacking and individual models
models = {
    "Equal Ensemble (Cat + LGBM + XGB + SGD)": "match_win_probA",
    "Stacking": "stacked_probA",
    "CatBoost": "catboost_probA",
    "LightGBM": "lightgbm_probA",
    "XGBoost": "xgboost_probA",
    "SGD": "sgd_probA"
}

results = []

for name, col in models.items():
    if col not in df.columns:
        continue

    y_prob = df[col].to_numpy()
    assert np.isfinite(y_prob).all()
    assert ((y_prob >= 0) & (y_prob <= 1)).all()

    y_pred = (y_prob >= 0.5).astype(int)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "Brier Score": brier_score_loss(y_true, y_prob),
        "Log Loss": log_loss(y_true, y_prob)
    })

results_df = pd.DataFrame(results)

print(f"Test matches: {len(df):,}")
print(results_df.to_string(index=False, float_format="%.5f"))

# Per-match winning probabilities and actual outcomes
df["predicted_winner"] = (df["match_win_probA"] >= 0.5).astype(int)

display_cols = [
    c for c in ["dataset_row", "match_id", "playerA_id", "playerB_id"]
    if c in df.columns
] + ["actual_winner", "match_win_probA", "predicted_winner"]

display(df[display_cols].head(20))

# Save evaluation results
results_df.to_csv(
    V2 / "predictions/v2_1_test_metrics.csv",
    index=False
)


Test matches: 51,666
                                  Model  Accuracy  ROC-AUC  Brier Score  Log Loss
Equal Ensemble (Cat + LGBM + XGB + SGD)   0.69047  0.76048      0.19887   0.58105
                               Stacking   0.69003  0.76037      0.19939   0.58249
                               CatBoost   0.69043  0.76004      0.19906   0.58139
                               LightGBM   0.68894  0.76009      0.19942   0.58309
                                XGBoost   0.68949  0.76050      0.19889   0.58114
                                    SGD   0.68536  0.75630      0.20042   0.58481


,dataset_row,match_id,playerA_id,playerB_id,actual_winner,match_win_probA,predicted_winner
0,207741,792958,106162,210200,0,0.656051,1
1,207742,792959,202418,126630,1,0.828779,1
2,207743,792960,208819,144972,1,0.392647,0
3,207744,792961,209863,138821,0,0.396240,0
4,207745,792962,123819,210167,0,0.570432,1
5,207746,792963,106072,209518,1,0.640937,1
6,207747,792964,202359,200276,1,0.876283,1
7,207748,792965,209322,212309,0,0.595330,1
8,207749,792966,144970,126107,0,0.298187,0
9,207750,792967,208518,209887,0,0.621544,1


In [90]:
# Test features are loaded only after tuning, stacking and all final-model fits.
import pandas as pd

test = pd.read_parquet(BASE_PATH / "v1_1_test.parquet")
test_log = pd.read_parquet(BASE_PATH / "v1_1_test_log.parquet")
test_ids = metadata_by_split["test"].reset_index(drop=True)
if len(test) != len(test_log) or len(test) != len(test_ids):
    raise ValueError("V1.1 test Parquet/metadata lengths differ")
if len(test) != manifest["split_sizes"]["test"]:
    raise ValueError("Test row count differs from V1.1 manifest")
if not (test.columns.equals(test_log.columns) and
        test.drop(columns=TARGETS_V1).columns.equals(X_train.columns)):
    raise ValueError("Test feature schema/order differs from training")
if not test[TARGETS_V1].reset_index(drop=True).equals(test_log[TARGETS_V1].reset_index(drop=True)):
    raise ValueError("Test and log data have inconsistent target row order")
if not test_ids["playerA_wins"].isin([0, 1]).all():
    raise ValueError("Missing or invalid V1.1 test match outcomes")
if not pd.to_datetime(test_ids["tourney_date"]).is_monotonic_increasing:
    raise ValueError("Test matches must remain in V1.1 chronological order")
X_test = test.drop(columns=TARGETS_V1).reset_index(drop=True)
X_test_log = test_log.drop(columns=TARGETS_V1).reset_index(drop=True)
del test, test_log


In [91]:
# Load the four already-refitted checkpoints. Do NOT train the models twice.
import joblib
import numpy as np
import gc

final_models = {
    "catboost": ("catboost_final_v2_1.pkl", False),
    "lightgbm": ("lightgbm_final_v2_1.pkl", False),
    "xgboost": ("xgboost_final_v2_1.pkl", False),
    "sgd": ("sgd_final_v2_1.pkl", True),
}
test_probabilities = {}
for name, (checkpoint, uses_log) in final_models.items():
    model_path = PROJECT_DIR / "models/final_models" / checkpoint
    if not model_path.exists():
        raise FileNotFoundError(f"Missing final {name} checkpoint: run its train + val cell first: {model_path}")
    model = joblib.load(model_path)
    X_eval = X_test_log if uses_log else X_test
    proba = np.asarray(model.predict_proba(X_eval))[:, 1]
    if len(proba) != len(test_ids) or not (
        np.isfinite(proba).all() and ((0 <= proba) & (proba <= 1)).all()
    ):
        raise ValueError(f"{name}: invalid or misaligned test probabilities")
    test_probabilities[name] = proba
    del model
    gc.collect()
    print(f"{name}: {len(proba):,} aligned test probabilities")


catboost: 51,666 aligned test probabilities
lightgbm: 51,666 aligned test probabilities
xgboost: 51,666 aligned test probabilities
sgd: 51,666 aligned test probabilities


In [92]:
# Primary prediction: 0.25 each CatBoost, LightGBM, XGBoost and SGD.
# Stacked probability, when trained, is an additional output, not the primary model.
import numpy as np
import json
import pandas as pd
import joblib

p_four = np.column_stack([test_probabilities[c] for c in validation_columns])
primary_prob = p_four @ ensemble_weights

test_predictions = pd.DataFrame({
    "dataset_row": test_ids["dataset_row"].to_numpy(),
    "match_id": test_ids["match_id"].to_numpy() if "match_id" in test_ids else None,
    "playerA_id": test_ids["playerA_id"].to_numpy() if "playerA_id" in test_ids else None,
    "playerB_id": test_ids["playerB_id"].to_numpy() if "playerB_id" in test_ids else None,
    "playerA_wins": test_ids["playerA_wins"].astype(int).to_numpy(),
    "match_win_probA": primary_prob,
    **{f"{name}_probA": prob for name, prob in test_probabilities.items()},
})

# The stacking meta-model is fitted in the validation section only (never on test).
stack_model_path = PROJECT_DIR / "models/ensembles/stack_xgb_lgbm_cat_sgd_v2_1.pkl"
if stack_model_path.exists():
    saved_stack = joblib.load(stack_model_path)
    X_stack_test = pd.DataFrame({name: test_probabilities[name] for name in stack_cols})
    test_predictions["stacked_probA"] = saved_stack.predict_proba(X_stack_test)[:, 1]

if not test_predictions["dataset_row"].is_unique:
    raise ValueError("Duplicate V1.1 test dataset rows")
if len(test_predictions) != len(metadata_by_split["test"]):
    raise ValueError("Missing test probability rows")
prob_cols = ["match_win_probA"] + [c for c in test_predictions if c.endswith("_probA")]
if not (np.isfinite(test_predictions[prob_cols].to_numpy()).all() and
        test_predictions[prob_cols].ge(0).all().all() and
        test_predictions[prob_cols].le(1).all().all()):
    raise ValueError("Test probabilities must lie in [0, 1]")
output_path = PROJECT_DIR / "predictions/v2_1_test_match_probabilities.parquet"
test_predictions.to_parquet(output_path, index=False)
print("Saved:", output_path, "| matches:", len(test_predictions))

prediction_manifest = {
    "source_dataset": str(BASE_PATH),
    "source_manifest_split_sizes": manifest["split_sizes"],
    "prediction_file": str(output_path),
    "match_identity": "dataset_row (V1.1 filtered-row ordinal)",
    "probability_direction": "P(Player A wins match)",
    "primary_probability_column": "match_win_probA",
    "primary_ensemble": "equal CatBoost + LightGBM + XGBoost + SGD",
    "primary_ensemble_weights": dict(zip(validation_columns, ensemble_weights.tolist())),
    "additional_stacked_probability": "stacked_probA" if "stacked_probA" in test_predictions else None,
    "final_training": "V1.1 train + validation; test labels unused for fitting or selection",
}
(PROJECT_DIR / "predictions/v2_1_prediction_manifest.json").write_text(
    json.dumps(prediction_manifest, indent=2), encoding="utf-8")


Saved: /content/drive/MyDrive/Projects/Tennis/v2_1/predictions/v2_1_test_match_probabilities.parquet | matches: 51666


790